# 04 - Azure ML Job Submission

This notebook does not contain any training logic. Its only job is to package up `src/train.py` and submit it as a job to a GPU compute cluster on Azure ML.

The full workflow is:
1. Authenticate and connect to your Azure ML workspace
2. Reference the already-registered data asset (upload only needed once — see Cell 3 note)
3. Define a Docker environment containing all required Python packages
4. Define the training job (script + data + environment + compute) and submit it
5. Stream logs back from the remote machine in real time

All training logic, e.g. data loading, log1p transforms, AutoGluon fit, MLflow logging, lives in `src/train.py`, not here.

## 1. Connect to Azure ML workspace

`MLClient` is the entry point for all Azure ML SDK operations. `DefaultAzureCredential` automatically finds your credentials — on a local machine it uses the Azure CLI login (`az login`). The three workspace identifiers below come from environment variables stored in `../.env`.

In [ ]:
from azure.ai.ml import MLClient, command, Input
from azure.ai.ml.entities import Environment, Data
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from pathlib import Path
import os

loaded = load_dotenv(dotenv_path=Path("..") / ".env")
print("Loaded .env:", loaded)

In [ ]:
SUBSCRIPTION_ID = os.environ["AZURE_SUBSCRIPTION_ID"]
RESOURCE_GROUP  = os.environ["AZURE_RESOURCE_GROUP"]
WORKSPACE_NAME  = os.environ["AZURE_WORKSPACE_NAME"]

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
print("Connected to workspace:", ml_client.workspace_name)

## 2. Reference data asset

The data asset `pasture-biomass-data v7` is already registered in Azure ML from the initial upload.

This cell **fetches** the existing registration — it does NOT re-upload anything. Re-uploading the dataset costs ~AU$35 in bandwidth each time (the dataset is >100 MB). Only run the upload cell below if you genuinely need a new version with different data.

In [ ]:
# Fetch the already-registered asset -- no upload, no bandwidth cost.
data_asset = ml_client.data.get(name="pasture-biomass-data", version="7")
print(f"Data asset: {data_asset.name} v{data_asset.version}")

### ⚠️ Upload cell — only run if data has changed

Uploading costs ~AU$35 in bandwidth. Only run if you have genuinely new training data and need a new version. Increment the version number before running.

In [ ]:
# --- DO NOT RUN unless you have new data ---
# Uploading the dataset again costs ~AU$35 in Azure bandwidth charges.
#
# new_data_asset = Data(
#     path=str(Path("..") / "data"),
#     type=AssetTypes.URI_FOLDER,
#     name="pasture-biomass-data",
#     version="8",   # increment version
#     description="Raw images and processed CSVs for pasture biomass prediction",
# )
# data_asset = ml_client.data.create_or_update(new_data_asset)
# print(f"Data asset registered: {data_asset.name} v{data_asset.version}")

## 3. Define compute environment

The environment specifies exactly what software is installed on the remote machine when the job runs. We start from a Microsoft-provided CUDA base image (which has GPU drivers pre-installed) and add our Python dependencies via conda/pip.

This environment is registered and versioned in Azure ML — once built, subsequent jobs reuse the cached Docker image rather than rebuilding from scratch. The first build takes 10–15 minutes.

In [ ]:
env = Environment(
    name="autogluon-env-v3",
    description="AutoGluon multimodal with MLflow",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-cuda11.8-cudnn8-ubuntu22.04",
    conda_file={
        "name": "autogluon-env",
        "channels": ["defaults", "conda-forge"],
        "dependencies": [
            "python=3.11",
            "pip",
            "setuptools",
            {"pip": [
                "autogluon.multimodal",
                "mlflow",
                "scikit-learn",
            ]}
        ]
    }
)
env = ml_client.environments.create_or_update(env)
print(f"Environment registered: {env.name}")

## 4. Define and submit the training job

A `command` job tells Azure ML: run this shell command, on this compute target, using this environment, with these data inputs. The `${{inputs.data_path}}` placeholder is substituted at runtime with the mounted path to the registered data asset.

`--time_limit` passes the specified seconds per target to `train.py` via argparse.

`ml_client.jobs.create_or_update(job)` submits the job and returns immediately. The cluster spins up, runs the script, saves artefacts, then spins back down. You are billed only for the active compute time.

In [ ]:
job = command(
    code="../src",
    command="python train.py --data_path ${{inputs.data_path}} --time_limit 14400",
    inputs={
        "data_path": Input(
            type=AssetTypes.URI_FOLDER,
            path=f"azureml:{data_asset.name}:{data_asset.version}",
        )
    },
    environment=f"{env.name}@latest",
    compute="gpu-cluster",
    experiment_name="pasture-biomass",
    display_name="autogluon-log1p-gpu-v1",
)

returned_job = ml_client.jobs.create_or_update(job)
print(f"Job submitted: {returned_job.name}")
print(f"Studio URL: {returned_job.studio_url}")

## 5. Stream training logs

Streams stdout from the remote machine back to this notebook in real time. This cell blocks until the job completes or fails.

You can also monitor the job in Azure ML Studio via the URL printed in Cell 4 above. It is safe to skip this cell and monitor in Studio instead — the job continues running on Azure regardless.

In [ ]:
ml_client.jobs.stream(returned_job.name)

## 6. Backbone comparison sweep

Submit one job per backbone architecture using the standard 80/20 train/val split with a 1800-second (30 min) time limit per target — enough to compare architectures fairly without a full convergence run.

Architectures being compared against the default baseline:
- `swin_base_patch4_window7_224` — Swin Transformer Base, a hierarchical vision transformer that performs well on dense visual tasks
- `efficientnet_b4` — EfficientNet B4, an efficient CNN with strong general-purpose image representations

All three runs appear side by side in the MLflow `pasture-biomass` experiment. Compare per-target R² across runs to identify the best backbone before committing to a full-length training job.

**Note:** the default backbone baseline is already trained (run from section 4). Only the two new backbones are submitted here.

In [ ]:
# Backbones to compare. Keys are used as display names in Azure ML Studio.
BACKBONES = {
    "swin-base"      : "swin_base_patch4_window7_224",
    "efficientnet-b4": "efficientnet_b4",
}

backbone_jobs = {}

for display_name, backbone in BACKBONES.items():
    job = command(
        code="../src",
        command=(
            "python train.py"
            " --data_path ${{inputs.data_path}}"
            " --time_limit 1800"
            f" --backbone {backbone}"
        ),
        inputs={
            "data_path": Input(
                type=AssetTypes.URI_FOLDER,
                path=f"azureml:{data_asset.name}:{data_asset.version}",
            )
        },
        environment=f"{env.name}@latest",
        compute="gpu-cluster",
        experiment_name="pasture-biomass",
        display_name=f"autogluon-{display_name}-v1",
    )
    submitted = ml_client.jobs.create_or_update(job)
    backbone_jobs[display_name] = submitted.name
    print(f"Submitted [{display_name}]: {submitted.name}")
    print(f"  Studio: {submitted.studio_url}\n")

print("All backbone jobs submitted. Monitor progress in Azure ML Studio.")

## 7. Cross-validation

Submit a 5-fold cross-validation job using the default AutoGluon backbone. With only 357 samples, a single 80/20 split gives validation metrics that vary depending on which 72 samples end up in the held-out set. 5-fold CV trains on 5 different splits and averages the results, giving more reliable performance estimates.

Each fold trains 4 predictors (one per target) with a 900-second time limit. Total job time: 5 folds × 4 targets × ~15 min = ~5 hours.

MLflow logs both per-fold metrics (e.g. `Dry_Green_g_fold2_r2`) and aggregate summaries (e.g. `Dry_Green_g_cv_r2_mean`, `Dry_Green_g_cv_r2_std`). High std means the model is sensitive to which samples it trains on — expected with small datasets.

In [ ]:
cv_job = command(
    code="../src",
    command=(
        "python train.py"
        " --data_path ${{inputs.data_path}}"
        " --time_limit 900"
        " --n_folds 5"
    ),
    inputs={
        "data_path": Input(
            type=AssetTypes.URI_FOLDER,
            path=f"azureml:{data_asset.name}:{data_asset.version}",
        )
    },
    environment=f"{env.name}@latest",
    compute="gpu-cluster",
    experiment_name="pasture-biomass",
    display_name="autogluon-cv5fold-v1",
)

cv_returned = ml_client.jobs.create_or_update(cv_job)
print(f"CV job submitted: {cv_returned.name}")
print(f"Studio URL: {cv_returned.studio_url}")